# Data Collection and Preprocessing Pipeline
**Project:** Vesuvius Challenge - Ink Masking  
**Institution:** DHLAB, EPFL

This notebook documents the initial data collection and preprocessing steps required to build the foundational papyrus dataset. It includes automated web scraping for metadata (Duke University) and raw images (University of Oslo), automated license filtering, interactive manual region-of-interest (ROI) cropping, and lossless TIFF standardization.

**Note:** The final dataset harmonization and global ID assignment are handled by `lib/metadata_merger.py`.

In [ ]:
import os
import re
import csv
import json
import time
import shutil
import urllib3
import requests
import cv2
from PIL import Image
from bs4 import BeautifulSoup
from urllib.parse import urljoin

# Disable SSL warnings for external archival scraping
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# Global Session Configuration
session = requests.Session()
session.headers.update({'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'})
session.verify = False

## 1. Duke University Archive: Metadata Scraping
The raw images from the Duke archive are downloaded manually. This section parses the filenames of the downloaded images to generate potential archive URLs, probes the Duke database, and scrapes the HTML to build the initial metadata CSV.

In [ ]:
# --- DUKE CONFIGURATION ---
IMAGE_FOLDER_DUKE = "../data/raw/duke/images"
OUTPUT_CSV_DUKE = "../data/raw/duke/metadata.csv"

def generate_possible_urls(filename):
    """
    Parses the filename and generates an exhaustive list of possible Duke URLs.
    Accounts for hidden fragment letters (a, b) and inverted side logic (a-r vs r-a).

    Args:
        filename (str): The raw filename of the Duke image.

    Returns:
        tuple: A list of candidate URLs and the extracted inventory number.
    """
    match = re.match(r'^(\d+)([rv]?)(?:-(left|right|a|b|1|2))?', filename.lower())
    
    if not match:
        return [], None
    
    num = match.group(1)
    side = match.group(2) 
    mod = match.group(3)  
    
    if mod in ['left', '1']:
        mod = 'a'
    elif mod in ['right', '2']:
        mod = 'b'
        
    urls_to_try = []
    base = "https://library.duke.edu/papyrus/records/"
    
    if side and mod:
        urls_to_try.extend([
            f"{base}{num}{side}-{mod}.html",
            f"{base}{num}{mod}-{side}.html",
            f"{base}{num}-{mod}.html",
            f"{base}{num}{mod}.html"
        ])
        
    if side:
        urls_to_try.append(f"{base}{num}{side}.html")
        if not mod:
            urls_to_try.extend([
                f"{base}{num}a-{side}.html",
                f"{base}{num}{side}-a.html",
                f"{base}{num}a.html"
            ])

    urls_to_try.append(f"{base}{num}.html")
    
    # Deduplicate while preserving order of likelihood
    unique_urls = list(dict.fromkeys(urls_to_try))
    return unique_urls, num

def get_duke_metadata(urls_to_try, inv_number):
    """
    Probes a list of URLs and scrapes metadata from the first valid response.

    Args:
        urls_to_try (list): List of candidate URLs.
        inv_number (str): The inventory number of the papyrus.

    Returns:
        dict: Scraped metadata fields or an error record.
    """
    successful_url = None
    response = None
    
    for url in urls_to_try:
        resp = requests.get(url)
        if resp.status_code == 200:
            successful_url = url
            response = resp
            break
        time.sleep(0.1) 
            
    if not successful_url:
        return {"Inventory": inv_number, "Error": f"Record not found. First guess: {urls_to_try[0]}"}

    soup = BeautifulSoup(response.text, 'html.parser')
    text_content = soup.get_text()
    
    metadata = {"URL": successful_url, "Inventory": inv_number}
    
    try:
        if "Title:" in text_content:
            metadata["Title"] = text_content.split("Title:")[1].split("Subject:")[0].strip()
        if "Subject:" in text_content:
            metadata["Subject (Date)"] = text_content.split("Subject:")[1].split("Material:")[0].strip()
        if "Material:" in text_content:
            metadata["Material"] = text_content.split("Material:")[1].split("Note:")[0].strip()
        if "Note:" in text_content:
            note_chunk = text_content.split("Note:")[1][:500].strip()
            metadata["Note"] = " ".join(note_chunk.split())
    except Exception:
        metadata["Error"] = "Could not parse HTML fields properly"
        
    return metadata

def scrape_duke_archive():
    """Main execution loop for Duke metadata extraction."""
    print(f"[INFO] Scanning directory: {IMAGE_FOLDER_DUKE}")
    if not os.path.exists(IMAGE_FOLDER_DUKE):
        print("[WARNING] Duke image directory not found. Please setup paths.")
        return

    files = [f for f in os.listdir(IMAGE_FOLDER_DUKE) if f.endswith(('.tif', '.gif', '.jpg'))]
    all_metadata = []
    
    for filename in files:
        urls_to_try, inv_number = generate_possible_urls(filename)
        
        if urls_to_try:
            print(f"[INFO] Fetching metadata for P.Duk.inv. {inv_number}...")
            data = get_duke_metadata(urls_to_try, inv_number)
            data["Filename"] = filename
            all_metadata.append(data)
            time.sleep(0.3) 
        else:
            print(f"[WARNING] Skipping {filename}: Could not extract inventory number.")

    if all_metadata:
        keys = ["Filename", "Inventory", "Title", "Subject (Date)", "Material", "Note", "URL", "Error"]
        os.makedirs(os.path.dirname(OUTPUT_CSV_DUKE), exist_ok=True)
        with open(OUTPUT_CSV_DUKE, 'w', newline='', encoding='utf-8') as output_file:
            dict_writer = csv.DictWriter(output_file, fieldnames=keys, extrasaction='ignore')
            dict_writer.writeheader()
            dict_writer.writerows(all_metadata)
        print(f"[SUCCESS] Duke metadata saved to {OUTPUT_CSV_DUKE}")

# scrape_duke_archive() # Uncomment to run

## 2. University of Oslo Archive: Image & Metadata Scraping
Unlike the Duke dataset, the Oslo OPES platform requires scraping both the high-resolution images and their associated metadata simultaneously. This section safely iterates through the catalog records to build the dataset.

In [ ]:
# --- OSLO CONFIGURATION ---
BASE_URL_OSLO = "https://ub-baser.uio.no/opes/"
OSLO_DIR = "../data/raw/oslo"
OSLO_IMG_DIR = os.path.join(OSLO_DIR, "images")
OSLO_META_DIR = os.path.join(OSLO_DIR, "metadata")

os.makedirs(OSLO_IMG_DIR, exist_ok=True)
os.makedirs(OSLO_META_DIR, exist_ok=True)

def extract_license_info(soup, page_url):
    """
    Determines the copyright license securely based on available metadata.

    Args:
        soup (BeautifulSoup): Parsed HTML of the record page.
        page_url (str): The URL of the page (for fallback reference).

    Returns:
        str: The extracted or inferred license text.
    """
    cc_link = soup.find('a', href=lambda href: href and "creativecommons.org" in href)
    if cc_link:
        return f"Creative Commons (Link: {cc_link['href']})"
    
    rights_section = soup.find(string=lambda text: text and "license" in text.lower())
    if rights_section:
        return rights_section.parent.get_text(strip=True)
    
    papyri_link = soup.find('a', href=lambda href: href and "papyri.info" in href)
    if papyri_link:
        external_url = papyri_link['href']
        if external_url.startswith("//"):
            external_url = "https:" + external_url
        return f"Creative Commons Attribution 3.0 Unported (Verified via cross-reference: {external_url})"
    
    return f"PENDING VERIFICATION - Oslo Exclusive Manuscript (No cross-reference found on {page_url})"

def process_oslo_record(page_url, record_id):
    """
    Scrapes the OPES page, downloads the media file, and saves JSON metadata.

    Args:
        page_url (str): The target URL.
        record_id (int): The loop index representing the record ID.

    Returns:
        int: 1 if successful, 0 if failed or 404.
    """
    try:
        response = session.get(page_url)
        if response.status_code == 404:
            return 0
            
        response.raise_for_status()
        soup = BeautifulSoup(response.text, 'html.parser')
        papyrus_id = f"oslo_record_{record_id}" 

        img_url = None
        img_link = soup.find('a', href=lambda href: href and "ub-media.uio.no/OPES/jpg/" in href)
        if img_link:
            img_url = img_link['href']
        else:
            img_tag = soup.find('img', src=lambda src: src and "ub-media.uio.no" in src)
            if img_tag:
                img_url = img_tag['src']

        if not img_url:
            print(f"[WARNING] Record {record_id}: Page found, but image missing.")
            return 0

        original_img_name = img_url.split('/')[-1]
        img_filename = f"{papyrus_id}_{original_img_name}" 
        img_filepath = os.path.join(OSLO_IMG_DIR, img_filename)
        
        # Download image
        img_data = session.get(img_url).content
        with open(img_filepath, 'wb') as handler:
            handler.write(img_data)

        # Build JSON
        metadata = {
            "fragment_id": papyrus_id,
            "original_image_name": original_img_name,
            "collection_source": "Oslo Papyri Electronic System (OPES)",
            "record_url": page_url,
            "image_url": img_url,
            "image_license": extract_license_info(soup, page_url),
            "required_citation": f"P.Oslo inv. {original_img_name.split('.')[0]}, Oslo Papyri Electronic System (OPES), University of Oslo Library" 
        }

        meta_filepath = os.path.join(OSLO_META_DIR, f"{papyrus_id}.json")
        with open(meta_filepath, 'w', encoding='utf-8') as f:
            json.dump(metadata, f, indent=4, ensure_ascii=False)
            
        print(f"[INFO] Record {record_id}: {img_filename} downloaded successfully.")
        return 1

    except Exception as e:
        print(f"[ERROR] Failed on Record {record_id}: {e}")
        return 0

def scrape_oslo_archive(max_id=350):
    """Main execution loop for Oslo bulk download."""
    print("[INFO] Initiating bulk archival retrieval...")
    total_downloads = 0
    
    for current_id in range(1, max_id + 1):
        url = f"https://ub-baser.uio.no/opes/record/{current_id}"
        total_downloads += process_oslo_record(url, current_id)
        time.sleep(1) 
        
    print(f"[SUCCESS] {total_downloads} fragments retrieved.")

# scrape_oslo_archive() # Uncomment to run

## 3. Oslo Dataset Cleanup
To comply with academic distribution constraints, any fragments exclusive to the Oslo collection that lack a verifiable open-access cross-reference ("PENDING VERIFICATION") must be purged from the dataset before training.

In [ ]:
def clean_oslo_dataset():
    """Removes images and JSON metadata for manuscripts with unverified licenses."""
    print("[INFO] Initiating dataset compliance cleanup...")
    deleted_count = 0

    if not os.path.exists(OSLO_META_DIR):
        print("[WARNING] Metadata directory not found.")
        return

    for filename in os.listdir(OSLO_META_DIR):
        if filename.endswith(".json"):
            meta_path = os.path.join(OSLO_META_DIR, filename)
            
            with open(meta_path, "r", encoding="utf-8") as f:
                data = json.load(f)
                
            if "PENDING VERIFICATION" in data.get("image_license", ""):
                papyrus_id = data.get("fragment_id")
                img_original_name = data.get("original_image_name", "")
                
                img_filename = f"{papyrus_id}_{img_original_name}"
                img_path = os.path.join(OSLO_IMG_DIR, img_filename)
                
                if os.path.exists(img_path):
                    os.remove(img_path)
                    
                os.remove(meta_path)
                deleted_count += 1

    print(f"[SUCCESS] Cleanup complete. {deleted_count} restricted items removed.")

# clean_oslo_dataset() # Uncomment to run

## 4. Interactive ROI Cropping
**WARNING: Interactive Step** This cell launches a GUI window (`cv2.selectROI`) for manual cropping of the archival images. **Do not run this cell in a headless environment (e.g., remote server without X11) as it will freeze the execution.**

In [ ]:
OSLO_CROPPED_DIR = os.path.join(OSLO_DIR, "images_cropped")

def manual_roi_crop(input_dir, output_dir):
    """
    Opens an interactive OpenCV window to allow manual bounding box cropping.

    Instructions:
        - Draw a rectangle (Left Click + Drag).
        - Press ENTER or SPACE to confirm and move to the next image.
        - Press ENTER without drawing to skip (copies image as-is).
        - Press 'C' to clear the current bounding box.
        - Press 'Q' or ESC to quit the program cleanly.
    """
    os.makedirs(output_dir, exist_ok=True)
    files = [f for f in os.listdir(input_dir) if f.lower().endswith(('.jpg', '.png', '.tif'))]

    print("[INFO] Launching manual cropping interface...")

    for filename in files:
        input_path = os.path.join(input_dir, filename)
        output_path = os.path.join(output_dir, filename)

        if os.path.exists(output_path):
            continue

        img = cv2.imread(input_path)
        if img is None:
            continue

        h, w = img.shape[:2]
        max_dim = 900 
        scale = 1.0
        display_img = img.copy()
        
        if h > max_dim or w > max_dim:
            scale = max_dim / max(h, w)
            display_img = cv2.resize(img, (int(w * scale), int(h * scale)))

        window_name = f"Crop: {filename}"
        roi = cv2.selectROI(window_name, display_img, showCrosshair=True, fromCenter=False)
        cv2.destroyWindow(window_name)

        x_small, y_small, w_small, h_small = roi

        if w_small == 0 or h_small == 0:
            print(f"[INFO] {filename} : Skipped (Copied as-is).")
            shutil.copy(input_path, output_path)
            continue

        x = int(x_small / scale)
        y = int(y_small / scale)
        w_crop = int(w_small / scale)
        h_crop = int(h_small / scale)

        cropped_img = img[y:y+h_crop, x:x+w_crop]
        cv2.imwrite(output_path, cropped_img)
        print(f"[INFO] {filename} : Cropped successfully.")

    cv2.destroyAllWindows()
    print("[SUCCESS] Manual cropping session ended.")

# manual_roi_crop(OSLO_IMG_DIR, OSLO_CROPPED_DIR) # Uncomment to run

## 5. Lossless TIFF Standardization
Computer Vision pipelines require pixel-perfect ground truth data. JPEG compression introduces artifacts around ink strokes that destroy segmentation accuracy. This script converts all images to `TIFF` using LZW compression (which is mathematically lossless).

In [ ]:
OSLO_READY_DIR = os.path.join(OSLO_DIR, "images_ready")

def convert_to_tiff(input_path, output_path):
    """
    Converts a standard image format (JPG, GIF) to a lossless LZW TIFF.

    Args:
        input_path (str): Filepath to the raw image.
        output_path (str): Filepath for the saved TIFF.
    """
    try:
        with Image.open(input_path) as img:
            if img.mode in ('P', 'RGBA'):
                img = img.convert('RGB')
            img.save(output_path, format='TIFF', compression='tiff_lzw')
    except Exception as e:
        print(f"[ERROR] Failed to convert {input_path} : {e}")

def batch_convert_folder_to_tiff(input_folder, output_folder):
    """Iterates through a directory and converts all valid images to TIFF."""
    print(f"[INFO] Starting TIFF conversion for directory: {input_folder}")
    os.makedirs(output_folder, exist_ok=True)
    
    target_extensions = ('.jpg', '.jpeg', '.gif')
    counter = 0
    
    if not os.path.exists(input_folder):
        print("[WARNING] Input folder not found.")
        return

    for filename in os.listdir(input_folder):
        if filename.lower().endswith(target_extensions):
            input_file = os.path.join(input_folder, filename)
            
            base_name = os.path.splitext(filename)[0]
            output_file = os.path.join(output_folder, f"{base_name}.tif")
            
            convert_to_tiff(input_file, output_file)
            counter += 1
            
    print(f"[SUCCESS] Conversion complete. {counter} images saved to '{output_folder}'.")

# batch_convert_folder_to_tiff(OSLO_CROPPED_DIR, OSLO_READY_DIR) # Uncomment to run